In [1]:
import cv2, os
import pandas as pd

## Configuration

In [2]:
# Load the video
video_path = '/Users/cyrilmonette/Library/CloudStorage/SynologyDrive-data/22.12-23.02_actuation_OH/Images/high-fps/subset/hive5_rpi1_day-230109.mp4'
cap = cv2.VideoCapture(video_path)

# Check if the video opened successfully
if not cap.isOpened():
    print("Oops! We couldn't open the video.")

# Load csv file with timestamps
csv_path = '/Users/cyrilmonette/Library/CloudStorage/SynologyDrive-data/22.12-23.02_actuation_OH/Images/high-fps/subset/hive5_rpi1_day-230109.csv'
df = pd.read_csv(csv_path, header=0)
print(df.head())

target_folder = '/Users/cyrilmonette/Library/CloudStorage/SynologyDrive-data/22.12-23.02_actuation_OH/Images/high-fps/subset/extracted_frames/'

   frame in hive5_rpi1_day-230109.mp4               datetime_utc
0                                   0  2023-01-09 00:00:01+00:00
1                                   1  2023-01-09 00:00:05+00:00
2                                   2  2023-01-09 00:00:09+00:00
3                                   3  2023-01-09 00:00:13+00:00
4                                   4  2023-01-09 00:00:17+00:00


In [3]:
hive_nb = video_path.split('/')[-1][4]
rpi_nb = video_path.split('/')[-1][9]

# Make sure both are integers
assert hive_nb.isdigit(), "Hive number should be a digit"
assert rpi_nb.isdigit(), "RPI number should be a digit"
hive_nb = int(hive_nb)
rpi_nb = int(rpi_nb)

target_folder = video_path.rsplit('/', 1)[0] + f"/h{hive_nb}r{rpi_nb}/"
print(f"Target folder for extracted frames: {target_folder}")

# Convert df['datetime_utc'] to pd.Timestamp
df['datetime_utc'] = pd.to_datetime(df['datetime_utc'])
# localize to UTC and convert to local time
df['datetime_utc'] = df['datetime_utc'].dt.tz_convert('UTC')

# mkdir target_folder if it doesn't exist
os.makedirs(target_folder, exist_ok=True)
frame_count = 0
max_frames = 30000

for i in range(max_frames):
    # Read a frame from the video
    ret, frame = cap.read()

    if not ret:
        break  # We're done when there are no more frames

    # Save the frame as an image, and take the name from the csv file
    image_filename = f"hive{hive_nb}_rpi{rpi_nb}_" + df['datetime_utc'][i].strftime('%y%m%d-%H%M%SZ') + '.jpg'
    cv2.imwrite(target_folder + image_filename, frame)

    frame_count += 1

# Release the video capture object
cap.release()

Target folder for extracted frames: /Users/cyrilmonette/Library/CloudStorage/SynologyDrive-data/22.12-23.02_actuation_OH/Images/high-fps/subset/h5r1/
